In [ ]:
# 1. Clean Install (2026 Standard)
!pip install -qU --prefer-binary \
    langchain-openai \
    langchain-community \
    langchain-chroma \
    langchain-classic \
    pypdf \
    openpyxl \
    gradio \
    "pandas==2.2.2"

In [ ]:
# 2. Imports
import os
import gradio as gr
from google.colab import userdata, files
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader, CSVLoader, UnstructuredExcelLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
# 3. Setup API Key
try:
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    print("✅ System Ready: Environment and API Key loaded.")
except:
    print("❌ ERROR: Please add 'OPENAI_API_KEY' to your Colab Secrets (🔑).")

In [ ]:
# Global variable to store the "Knowledge Base"
rag_chain = None

def update_knowledge_base(file):
    global rag_chain
    if file is None: return "Waiting for file..."

    # 1. Load the specific file type
    path = file.name
    if path.endswith('.pdf'): loader = PyPDFLoader(path)
    elif path.endswith('.csv'): loader = CSVLoader(path)
    else: loader = UnstructuredExcelLoader(path)

    # 2. Split and Vectorize
    docs = loader.load()
    splits = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100).split_documents(docs)
    vectorstore = Chroma.from_documents(splits, OpenAIEmbeddings(model="text-embedding-3-small"))

    # 3. Create the Chain
    llm = ChatOpenAI(model="gpt-4o", temperature=0)
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Answer based ONLY on context: {context}"),
        ("human", "{input}"),
    ])

    rag_chain = create_retrieval_chain(
        vectorstore.as_retriever(),
        create_stuff_documents_chain(llm, prompt)
    )
    return f"✅ Knowledge base updated with: {os.path.basename(path)}"

# Gradio 5.x expects (message, history)
def chat_interface(message, history):
    if rag_chain is None:
        return "Please upload a document on the left first!"

    # In Gradio 5, 'message' is usually an object. We want the text:
    user_text = message if isinstance(message, str) else message["text"]

    response = rag_chain.invoke({"input": user_text})
    return response["answer"]

# --- Build UI ---
with gr.Blocks(theme=gr.themes.Soft()) as app:
    gr.Markdown("# 📂 Universal RAG Chatbot")
    with gr.Row():
        with gr.Column(scale=1):
            file_uploader = gr.File(label="Upload New Document", file_types=[".pdf", ".csv", ".xlsx"])
            status = gr.Textbox(label="System Status", interactive=False, value="Ready - Upload a file")
            file_uploader.change(update_knowledge_base, inputs=file_uploader, outputs=status)

        with gr.Column(scale=3):
            # REMOVED type="messages" to fix the TypeError
            gr.ChatInterface(chat_interface)

app.launch(share=True, debug=True)

In [ ]:
!git config --global user.email "jatinranga32@gmail.com"
!git config --global user.name "jatinranga-Projects"

In [ ]:
!git clone https://github.com/jatinranga-Projects/my_rag_app